# G07 · A2 — OCR engine comparison: Tesseract vs PaddleOCR vs TrOCR vs Donut

Sir's OCR policy names three example pretrained recognizers (Tesseract / TrOCR / Donut);
this notebook adds PaddleOCR and runs all four on the **same small page sample**, for A2
form **Section 3** ("options we compared… ≥1 reproduced published/pretrained method").

**Scope:** the sample, not the full corpus. Three of these are deep models — a full
1,034-page run across all four would burn hours of Kaggle GPU quota for a comparison table
that doesn't need it. If you want a specific engine over more pages later, that's a small
change to the loop, not a rewrite.

**Kaggle setup:**
1. Add Input → attach `cruelangelssprint/pierce-1890-figure-and-ocr-outputs` (needed for
   page selection, same as `kaggle_heldout_ocr.ipynb`).
2. **Accelerator: GPU (T4 x2 or P100)** — TrOCR/Donut are much faster on GPU; PaddleOCR
   stays on CPU (more reliable install than the GPU wheel for a 12-page sample).
3. Internet **ON** (downloads ~3 model checkpoints from Hugging Face + PyPI installs).

**Keeping this comparable to the held-out set:** if you've already run
`kaggle_heldout_ocr.ipynb`, paste its printed `selected n=... : [...]` list into
`SELECTED_PAGES` in the selection cell below so both notebooks score the *identical* pages.
Otherwise this notebook re-derives the same selection independently (deterministic, same
rule) and prints it for you to reuse the other way around.

**Donut caveat (read before trusting its output):** `donut-base`'s own pretraining task
(SynthDoG: "read all text on the page") is a generic reader, not a document-QA fine-tune —
that's what's used here. But its decoder has a max output length, so very text-dense pages
may come back truncated. Check the printed output for that page before treating a bad Donut
score as a fair result.


In [ ]:
# ── 1. Environment ────────────────────────────────────────────────────────────
import os, sys, subprocess, time, json, re

def sh(cmd):
    print("$", cmd)
    subprocess.run(cmd, shell=True, check=True)

if subprocess.run(["which", "tesseract"], capture_output=True).returncode != 0:
    sh("apt-get -qq update > /dev/null 2>&1 || true")
    sh("apt-get -qq install -y tesseract-ocr > /dev/null 2>&1")

%pip install -q "pymupdf>=1.25.5,<1.26" "opencv-python-headless>=4.10,<5.0" "pytesseract>=0.3.13,<0.4" "pydantic>=2.7,<3.0" "pydantic-settings>=2.2,<3.0" "pyyaml>=6.0,<7.0"
%pip install -q paddleocr paddlepaddle
%pip install -q -U "transformers>=4.40" sentencepiece pillow

REPO_URL = "https://github.com/smammahdi/doc-agent-G07.git"
PIN = "45c3fc3"
if not os.path.exists("/kaggle/working/repo"):
    sh(f"git clone -q {REPO_URL} /kaggle/working/repo")
sh(f"cd /kaggle/working/repo && git checkout -q {PIN} && git log --oneline -1")
sys.path.insert(0, "/kaggle/working/repo/src")

import torch, fitz, cv2, numpy as np  # noqa: E402
from PIL import Image  # noqa: E402
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print("device:", DEVICE, "| torch:", torch.__version__, "| pymupdf:", getattr(fitz, "__version__", None) or fitz.version[0])
if DEVICE == "cpu":
    print("WARNING: no GPU detected — TrOCR/Donut will be slow. Check accelerator setting.")


In [ ]:
# ── 2. PDF: reuse the same uploaded copy if present, else fetch+verify from IA ──
from pathlib import Path
import hashlib, urllib.request

CANDIDATE = Path("/kaggle/input/datasets/kmazd1110/dl-peoples-common-sense-med-advisor/"
                  "EN_The-Peoples-Common-Sense-Medical-Adviser.pdf")
EXPECTED_BYTES = 65311598
EXPECTED_SHA = "841b1feb55ff0aff5735c3aeb308eb52e217f91ae55c5d34e21feb6a640c8896"

def sha256(p, chunk=1 << 20):
    h = hashlib.sha256()
    with open(p, "rb") as f:
        for b in iter(lambda: f.read(chunk), b""):
            h.update(b)
    return h.hexdigest()

if CANDIDATE.exists():
    PDF = CANDIDATE
    print("using uploaded copy:", PDF)
else:
    RAW = Path("/kaggle/working/data/raw"); RAW.mkdir(parents=True, exist_ok=True)
    PDF = RAW / "pierce-peoples-common-sense-medical-adviser-1890.pdf"
    if not (PDF.exists() and PDF.stat().st_size == EXPECTED_BYTES and sha256(PDF) == EXPECTED_SHA):
        print("uploaded copy not found — downloading ~65MB from Internet Archive instead...")
        urllib.request.urlretrieve(
            "https://archive.org/download/peoplescommonsen00pier/peoplescommonsen00pier.pdf", PDF)
    assert sha256(PDF) == EXPECTED_SHA, "downloaded PDF failed hash verification"
    print("using downloaded+verified copy:", PDF)

doc = fitz.open(str(PDF))
print(f"pages: {len(doc)}")


In [ ]:
# ── 3. Team sidecars: auto-discover DocAI words + Chandra layout ──────────────
def _sniff_keys(path, limit=5):
    keys = set()
    try:
        with open(path, encoding="utf-8") as f:
            for i, line in enumerate(f):
                line = line.strip()
                if not line:
                    continue
                row = json.loads(line)
                if isinstance(row, dict):
                    keys |= set(row)
                if i >= limit:
                    break
    except Exception:
        return set()
    return keys

docai_path = chandra_path = None
docai_size = chandra_size = 0
for root, _, files in os.walk("/kaggle/input"):
    for name in files:
        if not name.lower().endswith((".jsonl", ".json")):
            continue
        p = os.path.join(root, name)
        k = _sniff_keys(p); s = os.path.getsize(p)
        if {"page_id", "text", "bbox_norm"} <= k and s > docai_size:
            docai_path, docai_size = p, s
        elif {"bbox", "page_box"} <= k and s > chandra_size:
            chandra_path, chandra_size = p, s

print("DocAI words JSONL :", docai_path)
print("Chandra layout    :", chandra_path)
assert docai_path, "Document AI words file not found — is the sidecar dataset attached?"

from doc_agent.vision import ocr as ocr_mod        # noqa: E402
from doc_agent.vision import layout as layout_mod  # noqa: E402
ref_words = ocr_mod._load_reference_words(Path(docai_path))
chandra = layout_mod._load_chandra(Path(chandra_path)) if chandra_path else {}
print(f"DocAI  : {sum(map(len, ref_words.values())):,} words on {len(ref_words):,} pages")
print(f"Chandra: {sum(map(len, chandra.values())):,} blocks on {len(chandra):,} pages")


In [ ]:
# ── 4. Same deterministic held-out selection as kaggle_heldout_ocr.ipynb ──────
census = []
for i in range(len(doc)):
    pg = doc[i]; r = pg.rect
    pid = f"p{i + 1:04d}"
    census.append(dict(
        pid=pid, idx=i, w=round(r.width), h=round(r.height),
        landscape=(r.width > r.height) or pg.rotation in (90, 270),
        words=len(ref_words.get(pid, [])),
        figures=sum(1 for b in chandra.get(pid, []) if b["kind"] == "figure"),
    ))

def _spread(lst, k):
    if k <= 0 or len(lst) <= k:
        return lst[:max(k, 0)]
    return [lst[round(j * (len(lst) - 1) / (k - 1))] for j in range(k)]

figs = sorted((c for c in census if c["figures"] > 0 and c["words"] > 0),
              key=lambda c: (-c["figures"], c["pid"]))
low = sorted((c for c in census if c["words"] < 20), key=lambda c: c["pid"])
land = sorted((c for c in census if c["landscape"]), key=lambda c: c["pid"])
body = [c for c in census if c["words"] >= 150 and c["figures"] == 0 and not c["landscape"]]

proposal = []
def _add(cands, cat):
    for c in cands:
        if all(p["pid"] != c["pid"] for p in proposal):
            proposal.append({**c, "cat": cat})

_add(figs[:2], "figure"); _add(low[:2], "low/zero-word"); _add(land[:1], "landscape")
_add(_spread(body, 12 - len(proposal)), "body")
proposal.sort(key=lambda c: c["pid"])

# If you already ran kaggle_heldout_ocr.ipynb, paste its printed page-id list here so
# both notebooks score the identical pages, e.g. SELECTED_PAGES = ["p0002", "p0341", ...]
SELECTED_PAGES = None
sel = SELECTED_PAGES or [c["pid"] for c in proposal]
cat = {c["pid"]: c["cat"] for c in proposal}
idx = {c["pid"]: c["idx"] for c in census}
print(f"selected n={len(sel)}: {sel}")


In [ ]:
# ── 5. Render selected pages (same as the loader: 300 DPI JPEG q80) ──────────
from doc_agent.ingest import loader  # noqa: E402

OUT = Path("/kaggle/working/out")
PAGES_DIR = OUT / "pages"; PAGES_DIR.mkdir(parents=True, exist_ok=True)
for eng in ("tesseract", "paddleocr", "trocr", "donut"):
    (OUT / eng).mkdir(parents=True, exist_ok=True)

for pid in sel:
    target = PAGES_DIR / f"{pid}.jpg"
    if not target.exists():
        loader._atomic_render(doc[idx[pid]], target, 300, 80)
print("rendered:", sorted(p.name for p in PAGES_DIR.iterdir()))


In [ ]:
# ── 6. Tesseract on the sample (repo default pipeline, fast — for the side-by-side) ──
from doc_agent import config as config_mod  # noqa: E402
from doc_agent.contracts import Page        # noqa: E402

cfg = config_mod.load("/kaggle/working/repo/configs/config.yaml")
cfg["page_images"] = {pid: str(PAGES_DIR / f"{pid}.jpg") for pid in sel}
pages = [Page(id=pid, image_path=cfg["page_images"][pid], doc_id="pierce-1890") for pid in sel]
regions_by_page = {pid: layout_mod.detect([p], cfg) for pid, p in zip(sel, pages, strict=True)}

reader = ocr_mod.Reader(cfg)
for pid in sel:
    texts = [reader.transcribe_region(r) for r in regions_by_page[pid]]
    (OUT / "tesseract" / f"{pid}.txt").write_text("\n\n".join(t for t in texts if t), encoding="utf-8")
print("tesseract: done,", len(sel), "pages")


In [ ]:
# ── 7. PaddleOCR on the sample (its own detector + recognizer, CPU) ──────────
from paddleocr import PaddleOCR  # noqa: E402

paddle = PaddleOCR(use_angle_cls=True, lang="en", show_log=False)
for pid in sel:
    t0 = time.time()
    result = paddle.ocr(str(PAGES_DIR / f"{pid}.jpg"), cls=True)
    lines = [line[1][0] for line in (result[0] or [])]  # reading-order top-to-bottom
    (OUT / "paddleocr" / f"{pid}.txt").write_text("\n".join(lines), encoding="utf-8")
    print(f"  {pid}: {len(lines)} lines in {time.time() - t0:.1f}s")
print("paddleocr: done,", len(sel), "pages")


In [ ]:
# ── 8. TrOCR on the sample (line crops from the SAME regions Tesseract used) ──
from transformers import TrOCRProcessor, VisionEncoderDecoderModel  # noqa: E402

trocr_processor = TrOCRProcessor.from_pretrained("microsoft/trocr-base-printed")
trocr_model = VisionEncoderDecoderModel.from_pretrained("microsoft/trocr-base-printed").to(DEVICE).eval()

def split_lines(gray_crop, min_ink_ratio=0.01, max_gap=3):
    """Row-ink projection to split a paragraph-level region into single text lines —
    same technique as vision/layout.py's row detection, applied at finer granularity."""
    _, binary = cv2.threshold(gray_crop, 0, 255, cv2.THRESH_BINARY_INV + cv2.THRESH_OTSU)
    row_ink = np.count_nonzero(binary, axis=1)
    min_ink = max(1, int(round(gray_crop.shape[1] * min_ink_ratio)))
    active = row_ink >= min_ink
    lines, start, gap = [], None, 0
    for y, is_active in enumerate(active):
        if is_active:
            if start is None:
                start = y
            gap = 0
        elif start is not None:
            gap += 1
            if gap > max_gap:
                lines.append((start, y - gap + 1)); start = None; gap = 0
    if start is not None:
        lines.append((start, gray_crop.shape[0]))
    return lines

@torch.no_grad()
def trocr_read(pil_crop):
    pixel_values = trocr_processor(images=pil_crop, return_tensors="pt").pixel_values.to(DEVICE)
    ids = trocr_model.generate(pixel_values, max_length=64)
    return trocr_processor.batch_decode(ids, skip_special_tokens=True)[0].strip()

for pid in sel:
    t0 = time.time()
    img_bgr = cv2.imread(str(PAGES_DIR / f"{pid}.jpg"), cv2.IMREAD_COLOR)
    img_gray = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2GRAY)
    region_texts = []
    for region in regions_by_page[pid]:
        x0, y0, x1, y1 = region.bbox
        gray_crop = img_gray[y0:y1, x0:x1]
        if gray_crop.size == 0:
            continue
        line_texts = []
        for ly0, ly1 in split_lines(gray_crop):
            if ly1 - ly0 < 6:  # skip slivers
                continue
            pil_line = Image.fromarray(cv2.cvtColor(img_bgr[y0 + ly0:y0 + ly1, x0:x1], cv2.COLOR_BGR2RGB))
            line_texts.append(trocr_read(pil_line))
        region_texts.append(" ".join(t for t in line_texts if t))
    (OUT / "trocr" / f"{pid}.txt").write_text("\n\n".join(t for t in region_texts if t), encoding="utf-8")
    print(f"  {pid}: {time.time() - t0:.1f}s")
print("trocr: done,", len(sel), "pages")


In [ ]:
# ── 9. Donut on the sample (whole page, SynthDoG "read the page" pretraining task) ──
from transformers import DonutProcessor, VisionEncoderDecoderModel as DonutVED  # noqa: E402

donut_processor = DonutProcessor.from_pretrained("naver-clova-ix/donut-base")
donut_model = DonutVED.from_pretrained("naver-clova-ix/donut-base").to(DEVICE).eval()

@torch.no_grad()
def donut_read(pil_page):
    task_prompt = "<s_synthdog>"
    decoder_input_ids = donut_processor.tokenizer(
        task_prompt, add_special_tokens=False, return_tensors="pt"
    ).input_ids.to(DEVICE)
    pixel_values = donut_processor(pil_page, return_tensors="pt").pixel_values.to(DEVICE)
    outputs = donut_model.generate(
        pixel_values, decoder_input_ids=decoder_input_ids,
        max_length=donut_model.decoder.config.max_position_embeddings,
        pad_token_id=donut_processor.tokenizer.pad_token_id,
        eos_token_id=donut_processor.tokenizer.eos_token_id,
        bad_words_ids=[[donut_processor.tokenizer.unk_token_id]],
        return_dict_in_generate=True,
    )
    seq = donut_processor.batch_decode(outputs.sequences)[0]
    seq = seq.replace(donut_processor.tokenizer.eos_token or "", "")
    seq = seq.replace(donut_processor.tokenizer.pad_token or "", "")
    return re.sub(r"<.*?>", "", seq).strip()  # strip any structure/task tokens

for pid in sel:
    t0 = time.time()
    pil_page = Image.open(str(PAGES_DIR / f"{pid}.jpg")).convert("RGB")
    text = donut_read(pil_page)
    (OUT / "donut" / f"{pid}.txt").write_text(text, encoding="utf-8")
    hit_limit = len(text) > 0 and len(donut_processor.tokenizer(text).input_ids) >= \
        donut_model.decoder.config.max_position_embeddings - 2
    flag = "  <-- likely truncated (hit max length)" if hit_limit else ""
    print(f"  {pid}: {time.time() - t0:.1f}s, {len(text)} chars{flag}")
print("donut: done,", len(sel), "pages")


In [ ]:
# ── 10. Optional: score all four against hand-verified labels ─────────────────
# Set to your labels.jsonl once kaggle_heldout_ocr.ipynb has produced one (upload it as a
# Kaggle dataset/input, or drop it in /kaggle/working and point LABELS_PATH at it).
LABELS_PATH = None  # e.g. "/kaggle/input/g07-heldout-labels/labels.jsonl"

def norm(s):
    return re.sub(r"\s+", " ", s).strip()

def lev(a, b):
    if len(a) < len(b):
        a, b = b, a
    prev = list(range(len(b) + 1))
    for i, ca in enumerate(a, 1):
        cur = [i]
        for j, cb in enumerate(b, 1):
            cur.append(min(prev[j] + 1, cur[-1] + 1, prev[j - 1] + (ca != cb)))
        prev = cur
    return prev[-1]

if LABELS_PATH is None:
    print("LABELS_PATH not set — skipping scoring. Raw transcriptions are saved per engine; "
          "score them later once labels.jsonl exists (reuse this cell's logic).")
else:
    refs = {r["page_id"]: r["text"] for r in
            (json.loads(l) for l in open(LABELS_PATH, encoding="utf-8"))}
    print(f"{'engine':10} {'pages':>6} {'micro CER':>10}")
    for eng in ("tesseract", "paddleocr", "trocr", "donut"):
        tc = te = 0
        for pid in sel:
            if pid not in refs:
                continue
            ref = norm(refs[pid])
            hyp = norm((OUT / eng / f"{pid}.txt").read_text(encoding="utf-8"))
            te += lev(hyp, ref); tc += len(ref)
        print(f"{eng:10} {len(sel):>6} {te / max(1, tc):>10.4f}")


In [ ]:
# ── 11. Package for download ───────────────────────────────────────────────────
import shutil
zip_path = shutil.make_archive("/kaggle/working/ocr_comparison_output", "zip", str(OUT))
print("DONE. Download from the right panel → Output: ocr_comparison_output.zip")
print("  contains: pages/, tesseract/, paddleocr/, trocr/, donut/  (one .txt per page each)")
print(f"selected pages ({len(sel)}):", sel)
